In [ ]:
from datetime import datetime
from pathlib import Path
import sys

sys.path.insert(0, str(Path("../..").resolve()))

import ad_safe_lib as ad_safe

print("challenge:", ad_safe.CHALLENGE_DIR)
print("device:", ad_safe.DEVICE)
print("dataset:", ad_safe.DATA_DIR)
examples_root = ad_safe.AD_SAFE_RUNS_DIR / "notebook_examples"
examples_root.mkdir(parents=True, exist_ok=True)


# Teacher Then Student Distillation

Train a small pretrained teacher, warm up the full scratch student, then continue that student with cached teacher logits. The run is still notebook-sized, but it uses enough data and epochs for the student to learn before distillation.


In [ ]:
run_id = "notebook-03-" + datetime.now().strftime("%Y-%m-%d-%H-%M-%S")
output_dir = examples_root / run_id

# Slightly larger subset reduces variance while staying notebook-friendly.
train_source = ad_safe.DatasetSourceSpec("train", fraction=0.08, seed=30)
eval_sources = {
    "val": ad_safe.DatasetSourceSpec("val", fraction=0.10, seed=101),
    "test": ad_safe.DatasetSourceSpec("test", fraction=0.10, seed=102),
}
teacher_path = output_dir / "teacher_mobilenet.pt"
student_seed = 32

teacher_phase = ad_safe.PhaseSpec(
    job_index=0,
    phase_index=0,
    prefix="teacher_mobilenet",
    title="teacher",
    requested_seed=31,
    config=ad_safe.TrainingConfig(
        base_model="mobilenet_v3_small",
        epochs=3,
        patience=2,
        batch_size=16,
        learning_rate=(1e-3,),
        resplit_runs=1,
    ),
    model_filename=teacher_path.name,
    history_filename="teacher_mobilenet_history.png",
    json_filename="teacher_mobilenet.json",
)

student_warmup_phase = ad_safe.PhaseSpec(
    job_index=1,
    phase_index=0,
    prefix="student_warmup",
    title="warmup",
    requested_seed=student_seed,
    config=ad_safe.TrainingConfig(
        base_model="simple_cnn",
        epochs=4,
        patience=2,
        batch_size=16,
        learning_rate=(6e-4,),
        resplit_runs=1,
    ),
    unfreeze_all=True,
    model_filename="student_warmup.pt",
    history_filename="student_warmup_history.png",
    json_filename="student_warmup.json",
)

distilled_student_phase = ad_safe.PhaseSpec(
    job_index=1,
    phase_index=1,
    prefix="student_distilled",
    title="distilled",
    requested_seed=student_seed,
    config=ad_safe.TrainingConfig(
        base_model="simple_cnn",
        epochs=6,
        patience=3,
        batch_size=16,
        learning_rate=(2e-4,),
        resplit_runs=1,
        teacher_model_path=str(teacher_path),
        distillation_alpha=0.15,
        distillation_temperature=3.0,
    ),
    unfreeze_all=True,
    model_filename="student_distilled.pt",
    history_filename="student_distilled_history.png",
    json_filename="student_distilled.json",
)

plan = ad_safe.RunPlan(
    output_dir=output_dir,
    run_id=run_id,
    train_split="train",
    eval_splits=("val", "test"),
    jobs=(
        ad_safe.JobSpec(job_index=0, job_id="teacher", title="teacher", backbone="mobilenet_v3_small", phases=(teacher_phase,)),
        ad_safe.JobSpec(job_index=1, job_id="student", title="student", backbone="simple_cnn", phases=(student_warmup_phase, distilled_student_phase)),
    ),
    resume=False,
    force=True,
    setup_path=output_dir / "setup.json",
    check_foreign_contract=False,
    train_source=train_source,
    eval_sources=eval_sources,
)

result = ad_safe.run_training_plan(plan)
[(phase.prefix, phase.model_path) for phase in result.phase_results]


In [ ]:
ad_safe.run_evaluation_plan(
    ad_safe.EvaluationPlan(
        models=tuple(
            ad_safe.ModelEvalSpec(path=phase.model_path, title=phase.prefix)
            for phase in result.phase_results
        ),
        datasets=(
            ad_safe.DatasetEvalSpec(name="val", batch_size=16, source=eval_sources["val"]),
            ad_safe.DatasetEvalSpec(name="test", batch_size=16, source=eval_sources["test"]),
        ),
        output_dir=output_dir,
        sort_key="acc_test",
        title="Teacher/student comparison",
    )
)
